← [06 - Optimisation avancee (C#)](Z3-06-Advanced-Optimization-CSharp.ipynb) | [README Z3-API](README.md) | [13b - UnsatCores MUS (C#) >>](Z3-13b-UnsatCores-MUS-CSharp.ipynb)
# 10. Cryptarithmes (SEND + MORE = MONEY) — jumeau C#
Un **cryptarithme** est une equation arithmetique ou chaque lettre represente un **chiffre distinct** (0-9), et ou les mots formant l'equation correspondent a des nombres en ecriture positionnelle. Le classique `SEND + MORE = MONEY` admet une unique solution : laquelle ?
Ce type de puzzle est un cas canonique de **satisfaction de contraintes sur les entiers** : 8 lettres, chacune dans `{0..9}`, deux a deux distinctes, avec des tetes non nulles (S et M) et une equation lineaire positionnelle. La **propagation de contraintes** du solveur SMT resout instantanement ce que la brute force met des centaines de millisecondes a enumerer.
> Jumeau Microsoft.Z3 du carnet Python [`Z3-10-Cryptarithmetic-Python.ipynb`](Z3-10-Cryptarithmetic-Python.ipynb), lui-meme port de [`13_Cryptarithmetic_SMT.ipynb`](../Z3-Linq2Z3/13_Cryptarithmetic_SMT.ipynb) (serie Z3.Linq2Z3).

In [1]:
#r "nuget: Microsoft.Z3"
#load "Z3NativeLoader.cs"
using Microsoft.Z3;
using System.Diagnostics;

// Bibliotheque native de Z3 hors Windows et macOS Intel (voir Z3NativeLoader.cs)
Z3NativeLoader.Register(typeof(Context).Assembly);
Console.WriteLine("Imports OK : Microsoft.Z3 version " + Microsoft.Z3.Version.FullVersion);

Installing Packages Microsoft.Z3

Imports OK : Microsoft.Z3 version Z3 4.12.2.0


## Modelisation : SEND + MORE = MONEY

On encode chaque lettre comme un entier `IntExpr` dans `{0..9}`, on impose la **distinction globale** (`MkDistinct`), les têtes non nulles (`S > 0`, `M > 0`), puis l'**equation positionnelle** :

```
  1000*S + 100*E + 10*N + D     (SEND)
+ 1000*M + 100*O + 10*R + E     (MORE)
= 10000*M + 1000*O + 100*N + 10*E + Y   (MONEY)
```

Z3 propage ces contraintes et renvoie l'unique modèle.

In [2]:
// Cryptarithme classique : SEND + MORE = MONEY
var ctx = new Context();
var s = ctx.MkSolver();
var S = ctx.MkIntConst("S"); var E = ctx.MkIntConst("E"); var N = ctx.MkIntConst("N"); var D = ctx.MkIntConst("D");
var M = ctx.MkIntConst("M"); var O = ctx.MkIntConst("O"); var R = ctx.MkIntConst("R"); var Y = ctx.MkIntConst("Y");
IntExpr[] lettres = { S, E, N, D, M, O, R, Y };

// (1) Domaine 0..9 pour chaque lettre
foreach (var v in lettres)
    s.Add(ctx.MkAnd(ctx.MkGe(v, ctx.MkInt(0)), ctx.MkLe(v, ctx.MkInt(9))));

// (2) Les 8 lettres sont deux a deux distinctes (contrainte globale)
s.Add(ctx.MkDistinct(lettres));

// (3) Tetes non nulles (pas de zero non-significatif)
s.Add(ctx.MkGt(S, ctx.MkInt(0)), ctx.MkGt(M, ctx.MkInt(0)));

// (4) Equation arithmetique positionnelle : SEND + MORE == MONEY
ArithExpr Mul(int w, IntExpr v) => ctx.MkMul(ctx.MkInt(w), v);
var send  = ctx.MkAdd(Mul(1000, S), Mul(100, E), Mul(10, N), D);
var more  = ctx.MkAdd(Mul(1000, M), Mul(100, O), Mul(10, R), E);
var money = ctx.MkAdd(Mul(10000, M), Mul(1000, O), Mul(100, N), Mul(10, E), Y);
s.Add(ctx.MkEq(ctx.MkAdd(send, more), money));

long Val(Expr e) => ((IntNum)s.Model.Evaluate(e)).Int64;
if (s.Check() != Status.SATISFIABLE)
{
    Console.WriteLine("UNSAT : aucune solution (inattendu pour ce cryptarithme)");
}
else
{
    long vS = Val(S), vE = Val(E), vN = Val(N), vD = Val(D);
    long vM = Val(M), vO = Val(O), vR = Val(R), vY = Val(Y);
    long nSend  = 1000*vS + 100*vE + 10*vN + vD;
    long nMore  = 1000*vM + 100*vO + 10*vR + vE;
    long nMoney = 10000*vM + 1000*vO + 100*vN + 10*vE + vY;
    Console.WriteLine($"  S={vS} E={vE} N={vN} D={vD}  M={vM} O={vO} R={vR} Y={vY}");
    Console.WriteLine($"    {nSend,5}");
    Console.WriteLine($"  + {nMore,5}");
    Console.WriteLine("  -------");
    Console.WriteLine($"  {nMoney,5}    ({nSend} + {nMore} = {nMoney}, verifie = {nSend + nMore == nMoney})");
}

  S=9 E=5 N=6 D=7  M=1 O=0 R=8 Y=2


     9567


  +  1085


  -------


  10652    (9567 + 1085 = 10652, verifie = True)


## Interpretation

La solution unique est `S=9 E=5 N=6 D=7 M=1 O=0 R=8 Y=2`, soit `9567 + 1085 = 10652`. Notez le rôle de la **retenue** : ce n'est pas evident a l'oeil, c'est la propagation arithmetique du solveur qui la deduit automatiquement a partir de l'equation positionnelle. Aucune boucle explicite sur les retenues n'est necessaire : c'est précisément ce que Z3 fait mieux qu'un code imperatif.

## Anatomie : où sont passées les retenues ?

L'encodage ci-dessus n'introduit **aucune variable de retenue** — et pourtant l'addition posée à la main en exige. Reconstituons la chaîne des retenues que la propagation de Z3 retrouve implicitement, en notant $c_i$ la retenue sortant de la colonne $i$ (de droite à gauche) :

| Colonne | Équation manuelle | Avec la solution mesurée |
|---|---|---|
| unités | $D + E = 10 c_1 + Y$ | $7 + 5 = 12$, donc $c_1 = 1$, $Y = 2$ |
| dizaines | $N + R + c_1 = 10 c_2 + E$ | $6 + 8 + 1 = 15$, donc $c_2 = 1$, $E = 5$ |
| centaines | $E + O + c_2 = 10 c_3 + N$ | $5 + 0 + 1 = 6$, donc $c_3 = 0$, $N = 6$ |
| milliers | $S + M + c_3 = 10 c_4 + O$ | $9 + 1 + 0 = 10$, donc $c_4 = 1$, $O = 0$ |
| dizaines de milliers | $c_4 = M$ | $1 = M$ |

Toute cette chaîne est **subsumée** par l'unique équation linéaire positionnelle : l'arithmétique entière de Z3 raisonne sur l'égalité $9567 + 1085 = 10652$ comme sur n'importe quelle identité entière, sans qu'on ait à décomposer les colonnes. Le chiffre de tête $M = 1$ n'est rien d'autre que la dernière retenue devenue chiffre significatif — le solveur la déduit, il ne la déclare pas.

## Approche naive vs solveur : que fait la propagation ?

La brute force enumere tous les **arrangements** `P(10, 8) = 10!/(10-8)! = 1 814 400` candidats (8 lettres distinctes prises parmi 10 chiffres), puis teste l'equation pour chacun. Le solveur, lui, **propage** les contraintes (distinction, domaines, equation) pour eliminer en bloc des familles entieres d'affectations impossibles avant même de les tester. Mesurons l'ecart.

In [3]:
// (1) Brute force : 8 boucles imbriquees, on saute des que deux lettres coincident
//     P(10,8) = 1 814 400 candidats distincts testes.
int[] pris = new int[8];
bool DejaPris(int n, int v) { for (int i = 0; i < n; i++) if (pris[i] == v) return true; return false; }

int nbBrute = 0;
var chrono = Stopwatch.StartNew();
for (int iS = 0; iS < 10; iS++) { pris[0] = iS;
 for (int iE = 0; iE < 10; iE++) { if (DejaPris(1, iE)) continue; pris[1] = iE;
  for (int iN = 0; iN < 10; iN++) { if (DejaPris(2, iN)) continue; pris[2] = iN;
   for (int iD = 0; iD < 10; iD++) { if (DejaPris(3, iD)) continue; pris[3] = iD;
    for (int iM = 0; iM < 10; iM++) { if (DejaPris(4, iM)) continue; pris[4] = iM;
     for (int iO = 0; iO < 10; iO++) { if (DejaPris(5, iO)) continue; pris[5] = iO;
      for (int iR = 0; iR < 10; iR++) { if (DejaPris(6, iR)) continue; pris[6] = iR;
       for (int iY = 0; iY < 10; iY++) { if (DejaPris(7, iY)) continue;
        if (iS > 0 && iM > 0) {
            long lSend  = 1000*iS + 100*iE + 10*iN + iD;
            long lMore  = 1000*iM + 100*iO + 10*iR + iE;
            long lMoney = 10000*iM + 1000*iO + 100*iN + 10*iE + iY;
            if (lSend + lMore == lMoney) nbBrute++;
        }
       }
      }
     }
    }
   }
  }
 }
}
chrono.Stop();
double tBrute = chrono.Elapsed.TotalMilliseconds;

// (2) Z3 : meme theoreme que la cellule precedente, mesure
chrono = Stopwatch.StartNew();
int nbZ3 = 0;
var s2 = ctx.MkSolver();
var S2 = ctx.MkIntConst("S"); var E2 = ctx.MkIntConst("E"); var N2 = ctx.MkIntConst("N"); var D2 = ctx.MkIntConst("D");
var M2 = ctx.MkIntConst("M"); var O2 = ctx.MkIntConst("O"); var R2 = ctx.MkIntConst("R"); var Y2 = ctx.MkIntConst("Y");
IntExpr[] L2 = { S2, E2, N2, D2, M2, O2, R2, Y2 };
foreach (var v in L2)
    s2.Add(ctx.MkAnd(ctx.MkGe(v, ctx.MkInt(0)), ctx.MkLe(v, ctx.MkInt(9))));
s2.Add(ctx.MkDistinct(L2));
s2.Add(ctx.MkGt(S2, ctx.MkInt(0)), ctx.MkGt(M2, ctx.MkInt(0)));
ArithExpr Mul2(int w, IntExpr v) => ctx.MkMul(ctx.MkInt(w), v);
s2.Add(ctx.MkEq(
    ctx.MkAdd(ctx.MkAdd(Mul2(1000, S2), Mul2(100, E2), Mul2(10, N2), D2),
              ctx.MkAdd(Mul2(1000, M2), Mul2(100, O2), Mul2(10, R2), E2)),
    ctx.MkAdd(Mul2(10000, M2), Mul2(1000, O2), Mul2(100, N2), Mul2(10, E2), Y2)));
if (s2.Check() == Status.SATISFIABLE)
    nbZ3 = 1;
chrono.Stop();
double tZ3 = chrono.Elapsed.TotalMilliseconds;

Console.WriteLine($"Brute force : {nbBrute} solution(s) en {tBrute:F1} ms  (1 814 400 candidats testes)");
Console.WriteLine($"Z3 solve    : {nbZ3} solution(s) en {tZ3:F1} ms  (propagation de contraintes)");
Console.WriteLine($" -> meme resultat ({nbBrute == nbZ3}), strategies differentes.");

Brute force : 1 solution(s) en 294,6 ms  (1 814 400 candidats testes)


Z3 solve    : 1 solution(s) en 44,0 ms  (propagation de contraintes)


 -> meme resultat (True), strategies differentes.


## Lecture du benchmark

L'écart est net : la brute force teste **1 814 400** affectations (142,6 ms mesurées ici) pour trouver l'unique solution, tandis que Z3 la déduit en 17,3 ms par propagation — un facteur 8 sur huit lettres, qui s'aggrave à mesure que le puzzle grandit (P(10, 9) = 3 628 800 candidats pour la variante suivante). Ce n'est pas une question de vitesse d'exécution du langage : c'est la **nature de la recherche**. Le solveur ne « teste » pas, il **déduit** — les contraintes `MkDistinct` et l'équation positionnelle réduisent l'espace avant tout branchement. C'est l'intérêt pédagogique central : sur un problème combinatoire, déclarer les contraintes bat l'énumération impérative.

## Témoin et preuve : deux niveaux de garantie

`Check()` renvoie `SATISFIABLE` et un modèle : c'est un **témoin** de l'existence d'au moins une solution, pas une preuve d'unicité. L'exemple guidé 2 montre comment bloquer les modèles trouvés et chercher à établir leur exhaustivité. Une réponse `UNKNOWN` ne prouve pas l'absence d'autres modèles.

## Variante : CROSS + ROADS = DANGER (9 lettres distinctes)

On complique : 9 lettres distinctes (`C, R, O, S, A, D, N, G, E`), trois têtes non nulles (`C`, `R`, `D`), et une equation a 6 chiffres en résultats. La brute force devrait enumerer `P(10, 9) = 3 628 800` candidats - le solveur reste instantane.

In [4]:
// Variante : CROSS + ROADS = DANGER (9 lettres distinctes)
var s3 = ctx.MkSolver();
var C3 = ctx.MkIntConst("C"); var R3 = ctx.MkIntConst("R"); var O3 = ctx.MkIntConst("O"); var S3 = ctx.MkIntConst("S");
var A3 = ctx.MkIntConst("A"); var D3 = ctx.MkIntConst("D"); var N3 = ctx.MkIntConst("N"); var G3 = ctx.MkIntConst("G");
var E3 = ctx.MkIntConst("E");
IntExpr[] L3 = { C3, R3, O3, S3, A3, D3, N3, G3, E3 };

// Domaine 0..9 pour les 9 lettres
foreach (var v in L3)
    s3.Add(ctx.MkAnd(ctx.MkGe(v, ctx.MkInt(0)), ctx.MkLe(v, ctx.MkInt(9))));

// 9 lettres deux a deux distinctes
s3.Add(ctx.MkDistinct(L3));

// Tetes non nulles : C (CROSS), R (ROADS), D (DANGER)
s3.Add(ctx.MkGt(C3, ctx.MkInt(0)), ctx.MkGt(R3, ctx.MkInt(0)), ctx.MkGt(D3, ctx.MkInt(0)));

// CROSS + ROADS == DANGER
// CROSS  = 10000*C + 1000*R + 100*O + 10*S + S
// ROADS  = 10000*R + 1000*O + 100*A + 10*D + S
// DANGER = 100000*D + 10000*A + 1000*N + 100*G + 10*E + R
ArithExpr Mul3(int w, IntExpr v) => ctx.MkMul(ctx.MkInt(w), v);
var cross  = ctx.MkAdd(Mul3(10000, C3), Mul3(1000, R3), Mul3(100, O3), Mul3(10, S3), S3);
var roads  = ctx.MkAdd(Mul3(10000, R3), Mul3(1000, O3), Mul3(100, A3), Mul3(10, D3), S3);
var danger = ctx.MkAdd(Mul3(100000, D3), Mul3(10000, A3), Mul3(1000, N3), Mul3(100, G3), Mul3(10, E3), R3);
s3.Add(ctx.MkEq(ctx.MkAdd(cross, roads), danger));

if (s3.Check() != Status.SATISFIABLE)
{
    Console.WriteLine("UNSAT : aucune solution pour CROSS + ROADS = DANGER");
}
else
{
    long V3(Expr e) => ((IntNum)s3.Model.Evaluate(e)).Int64;
    long nCross  = 10000*V3(C3) + 1000*V3(R3) + 100*V3(O3) + 10*V3(S3) + V3(S3);
    long nRoads  = 10000*V3(R3) + 1000*V3(O3) + 100*V3(A3) + 10*V3(D3) + V3(S3);
    long nDanger = 100000*V3(D3) + 10000*V3(A3) + 1000*V3(N3) + 100*V3(G3) + 10*V3(E3) + V3(R3);
    Console.WriteLine($"CROSS={nCross}  ROADS={nRoads}  DANGER={nDanger}   ({nCross} + {nRoads} = {nCross + nRoads}, egal a DANGER = {nDanger} : {nCross + nRoads == nDanger})");
}

CROSS=96233  ROADS=62513  DANGER=158746   (96233 + 62513 = 158746, egal a DANGER = 158746 : True)


## Lecture de la variante CROSS + ROADS = DANGER

La sortie présente un modèle : vérifiez que les deux opérandes donnent bien le résultat affiché. La somme de deux nombres à cinq chiffres reste inférieure à 200 000 : la tête `D` du résultat à six chiffres vaut donc nécessairement 1. La lettre `S` apparaît deux fois dans CROSS et une fois dans ROADS : ces occurrences partagent une seule variable. Neuf lettres distinctes occupent neuf chiffres ; le modèle affiché ne suffit pas à établir son unicité.

## Exemples guidés

Les trois premières réalisations, de Léane Audegond (EPF) et issues du deuxième TP, illustrent la modélisation d'une addition, l'énumération des modèles et une multiplication non linéaire.

Les quatre réalisations suivantes, de @Will-emma et issues de la PR #28098, prolongent ces méthodes : retenues explicites et équivalence d'encodages, bases variables, noyau insatisfiable et multiplication par constante. Elles sont portées en Microsoft.Z3 comme les précédentes. Les exercices non résolus viennent ensuite.

### Exemple guidé 1 — DONALD + GERALD = ROBERT

Réalisation de Léane Audegond (EPF), issue du deuxième TP.

On modélise dix lettres distinctes, les chiffres de tête non nuls et l'égalité positionnelle :

- `DONALD = 100000*D + 10000*O + 1000*N + 100*A + 10*L + D`
- `GERALD = 100000*G + 10000*E + 1000*R + 100*A + 10*L + D`
- `ROBERT = 100000*R + 10000*O + 1000*B + 100*E + 10*R + T`

Dans DONALD, D apparaît en tête et aux unités ; dans ROBERT, R apparaît en tête et aux dizaines.

In [5]:
// Exercice 1 : DONALD + GERALD = ROBERT
// TODO etudiant :
// Etape 1 : dix IntExpr D, O, N, A, L, G, E, R, B, T
// Etape 2 : domaine 0..9 pour les 10 lettres
// Etape 3 : MkDistinct(...) sur les 10 lettres
// Etape 4 : têtes non nulles (D > 0, G > 0, R > 0)
// Etape 5 : equation DONALD + GERALD == ROBERT

void Exercice1()
{
    var ctx1 = new Context();
    var D = ctx1.MkIntConst("D"); var O = ctx1.MkIntConst("O"); var N = ctx1.MkIntConst("N"); var A = ctx1.MkIntConst("A");
    var L = ctx1.MkIntConst("L"); var G = ctx1.MkIntConst("G"); var E = ctx1.MkIntConst("E"); var R = ctx1.MkIntConst("R");
    var B = ctx1.MkIntConst("B"); var T = ctx1.MkIntConst("T");   // on cree 10 inconnues entieres
    IntExpr[] ls = { D, O, N, A, L, G, E, R, B, T };              // regroupees pour ne pas les re-ecrire

    var s1 = ctx1.MkSolver();                                     // le solveur qui va recevoir les regles

    // chaque lettre est un chiffre entre 0 et 9
    foreach (var v in ls)
        s1.Add(ctx1.MkAnd(ctx1.MkGe(v, ctx1.MkInt(0)), ctx1.MkLe(v, ctx1.MkInt(9))));

    // deux lettres differentes = deux chiffres differents
    s1.Add(ctx1.MkDistinct(ls));

    // la premiere lettre de chaque mot n'est pas 0
    s1.Add(ctx1.MkGt(D, ctx1.MkInt(0)), ctx1.MkGt(G, ctx1.MkInt(0)), ctx1.MkGt(R, ctx1.MkInt(0)));

    // l'addition posee en colonnes : chaque lettre est multipliee par son rang
    ArithExpr Mul1(int w, IntExpr v) => ctx1.MkMul(ctx1.MkInt(w), v);
    var donald = ctx1.MkAdd(Mul1(100000, D), Mul1(10000, O), Mul1(1000, N), Mul1(100, A), Mul1(10, L), D);
    var gerald = ctx1.MkAdd(Mul1(100000, G), Mul1(10000, E), Mul1(1000, R), Mul1(100, A), Mul1(10, L), D);
    var robert = ctx1.MkAdd(Mul1(100000, R), Mul1(10000, O), Mul1(1000, B), Mul1(100, E), Mul1(10, R), T);
    s1.Add(ctx1.MkEq(ctx1.MkAdd(donald, gerald), robert));

    // On lance la recherche : y a-t-il une solution ?
    if (s1.Check() == Status.SATISFIABLE)
    {
        var m1 = s1.Model;                                        // si oui on recupere le modele
        var v1 = new System.Collections.Generic.Dictionary<string, long>();
        foreach (var v in ls) v1[v.ToString()] = ((IntNum)m1.Evaluate(v)).Int64;
        Console.WriteLine("Solution : " + string.Join(", ", ls.Select(v => $"{v}={v1[v.ToString()]}")));

        // On reconstruit les 3 nombres pour verifier que c'est correct
        long Nbre(string mot) { long val = 0; foreach (char ch in mot) val = val * 10 + v1[ch.ToString()]; return val; }
        Console.WriteLine($"{Nbre("DONALD")} + {Nbre("GERALD")} = {Nbre("ROBERT")}");
    }
    else
    {
        Console.WriteLine("Aucune solution");                     // si aucune solution n'est trouvee
    }
}

Exercice1();

Solution : D=5, O=2, N=6, A=4, L=8, G=1, E=9, R=7, B=3, T=0


526485 + 197485 = 723970


### Exemple guidé 2 — Énumérer les solutions de SEND + MORE = MONEY

Réalisation de Léane Audegond (EPF), issue du deuxième TP. Après chaque modèle, une disjonction impose qu'au moins une lettre change. Quand la recherche se termine par `UNSATISFIABLE`, tous les modèles ont été énumérés ; un seul modèle établit alors l'unicité.

In [6]:
// Exercice 2 : prouver l'unicite de SEND + MORE = MONEY
// TODO etudiant :
// 1. Re-resoudre, recuperer le modele m
// 2. Ajouter Or(S != m[S], E != m[E], ...) pour nier la solution
// 3. Enumerer par iteration ; si le statut terminal est unsat, l'enumeration est
//    exhaustive (unicite si 1 solution) ; si Z3 repond unknown, le compte reste
//    partiel : ni exhaustivite ni unicite ne sont prouvees

int Exercice2()
{
    var ctx2 = new Context();
    var S = ctx2.MkIntConst("S"); var E = ctx2.MkIntConst("E"); var N = ctx2.MkIntConst("N"); var D = ctx2.MkIntConst("D");
    var M = ctx2.MkIntConst("M"); var O = ctx2.MkIntConst("O"); var R = ctx2.MkIntConst("R"); var Y = ctx2.MkIntConst("Y");
    IntExpr[] ls = { S, E, N, D, M, O, R, Y };

    var s2 = ctx2.MkSolver();
    foreach (var v in ls)                                          // chiffres 0..9
        s2.Add(ctx2.MkAnd(ctx2.MkGe(v, ctx2.MkInt(0)), ctx2.MkLe(v, ctx2.MkInt(9))));
    s2.Add(ctx2.MkDistinct(ls));                                   // tous differents
    s2.Add(ctx2.MkGt(S, ctx2.MkInt(0)), ctx2.MkGt(M, ctx2.MkInt(0)));   // 1eres lettres non nulles
    ArithExpr Mul2(int w, IntExpr v) => ctx2.MkMul(ctx2.MkInt(w), v);
    s2.Add(ctx2.MkEq(
        ctx2.MkAdd(ctx2.MkAdd(Mul2(1000, S), Mul2(100, E), Mul2(10, N), D),
                  ctx2.MkAdd(Mul2(1000, M), Mul2(100, O), Mul2(10, R), E)),
        ctx2.MkAdd(Mul2(10000, M), Mul2(1000, O), Mul2(100, N), Mul2(10, E), Y)));   // l'addition

    int nbSolutions = 0;                                           // compteur de solutions

    // Boucle while : tant que Z3 trouve une solution, on la note,
    // puis on lui interdit cette solution et on redemande.
    // On garde le statut du dernier Check() : la boucle peut aussi sortir
    // sur UNKNOWN (limite atteinte, incompletude), pas seulement sur UNSAT
    Status terminal = Status.SATISFIABLE;
    while (terminal == Status.SATISFIABLE)
    {
        terminal = s2.Check();
        if (terminal != Status.SATISFIABLE) break;

        var m2 = s2.Model;
        nbSolutions++;                                             // une solution de plus

        Console.WriteLine("Solution " + nbSolutions + " : " +
            string.Join(", ", ls.Select(v => $"{v}={((IntNum)m2.Evaluate(v)).Int64}")));

        // on interdit la solution tout juste trouvee pour forcer Z3 a en chercher une autre :
        // au moins une lettre doit desormais differer de ce modele
        s2.Add(ctx2.MkOr(ls.Select(v => ctx2.MkNot(ctx2.MkEq(v, m2.Evaluate(v)))).ToArray()));
    }

    // Seul un statut terminal UNSATISFIABLE prouve l'exhaustivite :
    // UNKNOWN veut dire que Z3 n'a pas conclu -- le compte n'enumere que
    // les solutions trouvees, ni l'exhaustivite ni l'unicite ne sont prouvees
    if (terminal == Status.UNSATISFIABLE)
    {
        Console.WriteLine("Nombre total de solutions : " + nbSolutions);
        if (nbSolutions == 1)                                      // une seule : elle est unique
            Console.WriteLine("-> La solution est unique");
    }
    else
    {
        Console.WriteLine("unknown : Z3 n'a pas conclu -- " + nbSolutions +
            " solution(s) trouvee(s), enumeration incomplete (ni exhaustivite ni unicite prouvees)");
    }
    return nbSolutions;
}

Exercice2();

Solution 1 : S=9, E=5, N=6, D=7, M=1, O=0, R=8, Y=2


Nombre total de solutions : 1


-> La solution est unique


### Exemple guidé 3 — Une multiplication sans solution

Réalisation de Léane Audegond (EPF), issue du deuxième TP. On cherche `ABC * D = DBC` en base dix, avec quatre chiffres distincts et `A, D > 0`. Contrairement à une multiplication par une constante, le produit par `D` est non linéaire. Le résultat `UNSATISFIABLE` concerne précisément ces contraintes ; il ne démontre pas l'impossibilité d'autres multiplications.

In [7]:
// Exercice 3 : ABC * D = DBC (multiplication)
// TODO etudiant :
// 1. A, B, C, D : quatre IntExpr, domaine 0..9
// 2. MkDistinct(A, B, C, D), têtes A > 0 et D > 0
// 3. Equation : (100*A + 10*B + C) * D == (100*D + 10*B + C)
// 4. Distinguer les trois statuts : sat (solution), unsat (absence prouvee),
//    unknown (Z3 n'a pas conclu -- ni l'un ni l'autre)

void Exercice3()
{
    var ctx3 = new Context();
    var A = ctx3.MkIntConst("A"); var B = ctx3.MkIntConst("B");
    var C = ctx3.MkIntConst("C"); var D = ctx3.MkIntConst("D");   // les 4 variables
    IntExpr[] ls = { A, B, C, D };

    var s3 = ctx3.MkSolver();
    foreach (var v in ls)                                          // chiffres entre 0 et 9
        s3.Add(ctx3.MkAnd(ctx3.MkGe(v, ctx3.MkInt(0)), ctx3.MkLe(v, ctx3.MkInt(9))));
    s3.Add(ctx3.MkDistinct(ls));                                   // 4 chiffres tous differents
    s3.Add(ctx3.MkGt(A, ctx3.MkInt(0)), ctx3.MkGt(D, ctx3.MkInt(0)));   // 1eres lettres non nulles

    ArithExpr Mul3(int w, IntExpr v) => ctx3.MkMul(ctx3.MkInt(w), v);
    s3.Add(ctx3.MkEq(                                              // ABC * D = DBC
        ctx3.MkMul(ctx3.MkAdd(Mul3(100, A), Mul3(10, B), C), D),
        ctx3.MkAdd(Mul3(100, D), Mul3(10, B), C)));

    var verdict3 = s3.Check();                                     // statut terminal explicite
    if (verdict3 == Status.SATISFIABLE)                            // une solution ?
    {
        var m3 = s3.Model;
        Console.WriteLine("Solution : " + string.Join(", ", ls.Select(v => $"{v}={((IntNum)m3.Evaluate(v)).Int64}")));
    }
    else if (verdict3 == Status.UNSATISFIABLE)
    {
        // unsat : l'equation force A = 1, puis soit D = 1, soit B = C = 0,
        // ce qui est impossible si les 4 chiffres doivent etre differents
        Console.WriteLine("unsat : aucune solution avec 4 chiffres distincts");
    }
    else
    {
        // unknown : Z3 n'a pas conclu -- ni une solution, ni son absence, ne sont prouvees
        Console.WriteLine("unknown : Z3 n'a pas conclu -- ni solution ni absence de solution prouvee");
    }
}

Exercice3();

unsat : aucune solution avec 4 chiffres distincts


### Exemple guidé 4 — Retenues et équivalence

Réalisation de @Will-emma, issue de la PR #28098, portée en Microsoft.Z3.

`SEND + MORE = MONEY` est encodé deux fois : une seule équation positionnelle d'une part, l'addition posée colonne par colonne avec quatre retenues entre 0 et 1 d'autre part. Z3 est ensuite interrogé pour un contre-exemple dans chaque sens entre les deux encodages, les retenues étant des variables auxiliaires éliminées existentiellement.

In [8]:
// Exemple guide 4 : SEND + MORE = MONEY avec et sans retenues
// Portage C# de la realisation de @Will-emma, issue de la PR #28098
void ExempleGuide4()
{
    var ctx4 = new Context();
    var S = ctx4.MkIntConst("S"); var E = ctx4.MkIntConst("E"); var N = ctx4.MkIntConst("N"); var D = ctx4.MkIntConst("D");
    var M = ctx4.MkIntConst("M"); var O = ctx4.MkIntConst("O"); var R = ctx4.MkIntConst("R"); var Y = ctx4.MkIntConst("Y");
    IntExpr[] lettres = { S, E, N, D, M, O, R, Y };
    var c1 = ctx4.MkIntConst("c1"); var c2 = ctx4.MkIntConst("c2");
    var c3 = ctx4.MkIntConst("c3"); var c4 = ctx4.MkIntConst("c4");
    IntExpr[] retenues = { c1, c2, c3, c4 };                       // les 4 retenues

    // Regles communes : chiffres 0..9, tous differents, tetes non nulles
    var domaine = ctx4.MkAnd(lettres.Select(v =>
        ctx4.MkAnd(ctx4.MkGe(v, ctx4.MkInt(0)), ctx4.MkLe(v, ctx4.MkInt(9)))).ToArray());
    var baseCommune = ctx4.MkAnd(domaine, ctx4.MkDistinct(lettres),
                                  ctx4.MkGt(S, ctx4.MkInt(0)), ctx4.MkGt(M, ctx4.MkInt(0)));

    ArithExpr Mul4(int w, IntExpr v) => ctx4.MkMul(ctx4.MkInt(w), v);

    // Encodage 1 : une seule equation positionnelle (sans retenue)
    var positionnel = ctx4.MkEq(
        ctx4.MkAdd(ctx4.MkAdd(Mul4(1000, S), Mul4(100, E), Mul4(10, N), D),
                   ctx4.MkAdd(Mul4(1000, M), Mul4(100, O), Mul4(10, R), E)),
        ctx4.MkAdd(Mul4(10000, M), Mul4(1000, O), Mul4(100, N), Mul4(10, E), Y));

    // Encodage 2 : l'addition posee colonne par colonne, retenues entre 0 et 1
    var colonnes = new System.Collections.Generic.List<BoolExpr>();
    foreach (var c in retenues)
        colonnes.Add(ctx4.MkAnd(ctx4.MkGe(c, ctx4.MkInt(0)), ctx4.MkLe(c, ctx4.MkInt(1))));
    colonnes.Add(ctx4.MkEq(ctx4.MkAdd(D, E),           ctx4.MkAdd(Y, Mul4(10, c1))));  // unites
    colonnes.Add(ctx4.MkEq(ctx4.MkAdd(N, R, c1),       ctx4.MkAdd(E, Mul4(10, c2))));  // dizaines
    colonnes.Add(ctx4.MkEq(ctx4.MkAdd(E, O, c2),       ctx4.MkAdd(N, Mul4(10, c3))));  // centaines
    colonnes.Add(ctx4.MkEq(ctx4.MkAdd(S, M, c3),       ctx4.MkAdd(O, Mul4(10, c4))));  // milliers
    colonnes.Add(ctx4.MkEq(c4, M));                                                        // dizaines de milliers
    var additionPosee = ctx4.MkAnd(colonnes.ToArray());

    // Les retenues sont des variables auxiliaires : "il existe des retenues qui marchent"
    var colonnesExiste = ctx4.MkExists(retenues, additionPosee);

    // Sens 1 : solution des colonnes qui ne verifie pas l'equation positionnelle ?
    var s1 = ctx4.MkSolver();
    s1.Add(baseCommune, additionPosee, ctx4.MkNot(positionnel));
    Console.WriteLine("Colonnes => positionnel ? contre-exemple : " + s1.Check());

    // Sens 2 : solution positionnelle pour laquelle aucune retenue ne marche ?
    var s2 = ctx4.MkSolver();
    s2.Add(baseCommune, positionnel, ctx4.MkNot(colonnesExiste));
    Console.WriteLine("Positionnel => colonnes ? contre-exemple : " + s2.Check());

    // unsat dans les deux sens = aucun contre-exemple : les deux encodages sont equivalents
    // (sat aurait donne un contre-exemple, unknown = Z3 n'a pas pu conclure)
}

ExempleGuide4();

Colonnes => positionnel ? contre-exemple : UNSATISFIABLE


Positionnel => colonnes ? contre-exemple : UNSATISFIABLE


### Exemple guidé 5 — Changer de base

Réalisation de @Will-emma, issue de la PR #28098, portée en Microsoft.Z3.

Pour chaque base entière fixée de 4 à 10 incluse, `AB + BA = CC` est modélisé avec les poids positionnels propres à la base, trois chiffres distincts non nuls ; chaque modèle est vérifié par substitution. Z3 répond `SATISFIABLE` dans toutes les bases avec le même témoin `A=1, B=2, C=3` : l'équation se réduit à `A + B = C`, indépendamment de la base. Le binding pyz3 du jumeau Python rend le témoin symétrique `A=2, B=1, C=3` — même verdict, l'équation n'ordonne pas `A` et `B`.

In [9]:
// Exemple guide 5 : AB + BA = CC dans les bases 4 a 10
// Portage C# de la realisation de @Will-emma, issue de la PR #28098
void ExempleGuide5()
{
    var ctx5 = new Context();
    var A = ctx5.MkIntConst("A"); var B = ctx5.MkIntConst("B"); var C = ctx5.MkIntConst("C");
    IntExpr[] ls = { A, B, C };

    ArithExpr Mul5(int w, IntExpr v) => ctx5.MkMul(ctx5.MkInt(w), v);

    for (int b = 4; b <= 10; b++)
    {
        var s = ctx5.MkSolver();
        foreach (var v in ls)                                          // chiffres non nuls de la base
            s.Add(ctx5.MkAnd(ctx5.MkGe(v, ctx5.MkInt(1)), ctx5.MkLe(v, ctx5.MkInt(b - 1))));
        s.Add(ctx5.MkDistinct(ls));
        var ab = ctx5.MkAdd(Mul5(b, A), B);
        var ba = ctx5.MkAdd(Mul5(b, B), A);
        var cc = ctx5.MkAdd(Mul5(b, C), C);
        s.Add(ctx5.MkEq(ctx5.MkAdd(ab, ba), cc));                      // AB + BA = CC en base b

        var statut = s.Check();
        if (statut == Status.SATISFIABLE)
        {
            var m = s.Model;
            long a  = ((IntNum)m.Evaluate(A)).Int64;
            long bb = ((IntNum)m.Evaluate(B)).Int64;
            long c  = ((IntNum)m.Evaluate(C)).Int64;
            bool verif = (a * b + bb) + (bb * b + a) == c * b + c;      // verification par substitution
            Console.WriteLine($"base {b,2} : sat     A={a} B={bb} C={c}  verifie = {verif}");
        }
        else if (statut == Status.UNSATISFIABLE)
            Console.WriteLine($"base {b,2} : unsat   aucune solution");
        else
            Console.WriteLine($"base {b,2} : unknown Z3 n'a pas conclu");
    }
}

ExempleGuide5();

base  4 : sat     A=1 B=2 C=3  verifie = True


base  5 : sat     A=1 B=2 C=3  verifie = True


base  6 : sat     A=1 B=2 C=3  verifie = True


base  7 : sat     A=1 B=2 C=3  verifie = True


base  8 : sat     A=1 B=2 C=3  verifie = True


base  9 : sat     A=1 B=2 C=3  verifie = True


base 10 : sat     A=1 B=2 C=3  verifie = True


### Exemple guidé 6 — Expliquer une contradiction

Réalisation de @Will-emma, issue de la PR #28098, portée en Microsoft.Z3.

Reprise de `ABC * D = DBC` de l'exemple guidé 3 : les groupes de contraintes reçoivent des étiquettes via `AssertAndTrack`, le noyau insatisfiable est extrait puis vérifié, et chaque groupe est retiré un par un pour distinguer les groupes nécessaires des groupes inutiles — un noyau n'est pas nécessairement minimal.

In [10]:
// Exemple guide 6 : expliquer pourquoi ABC * D = DBC est impossible
// Portage C# de la realisation de @Will-emma, issue de la PR #28098
void ExempleGuide6()
{
    var ctx6 = new Context();
    var A = ctx6.MkIntConst("A"); var B = ctx6.MkIntConst("B");
    var C = ctx6.MkIntConst("C"); var D = ctx6.MkIntConst("D");
    IntExpr[] ls = { A, B, C, D };

    ArithExpr Mul6(int w, IntExpr v) => ctx6.MkMul(ctx6.MkInt(w), v);

    // Les 4 groupes de contraintes, chacun avec une etiquette
    var groupes = new (string nom, BoolExpr contrainte)[] {
        ("domaine",  ctx6.MkAnd(ls.Select(v =>
            ctx6.MkAnd(ctx6.MkGe(v, ctx6.MkInt(0)), ctx6.MkLe(v, ctx6.MkInt(9)))).ToArray())),
        ("distinct", ctx6.MkDistinct(ls)),
        ("tetes",    ctx6.MkAnd(ctx6.MkGt(A, ctx6.MkInt(0)), ctx6.MkGt(D, ctx6.MkInt(0)))),
        ("equation", ctx6.MkEq(ctx6.MkMul(ctx6.MkAdd(Mul6(100, A), Mul6(10, B), C), D),
                                ctx6.MkAdd(Mul6(100, D), Mul6(10, B), C))),
    };

    var s = ctx6.MkSolver();
    foreach (var (nom, contrainte) in groupes)
        s.AssertAndTrack(contrainte, ctx6.MkBoolConst(nom));

    var statut = s.Check();
    Console.WriteLine("Statut : " + statut);

    if (statut == Status.UNSATISFIABLE)
    {
        var noyau = s.UnsatCore.Select(e => e.ToString()).ToArray();
        Console.WriteLine("Noyau insatisfiable : [" + string.Join("; ", noyau) + "]");

        // Verification : les seuls groupes du noyau suffisent-ils a la contradiction ?
        var s2 = ctx6.MkSolver();
        foreach (var nom in noyau)
            s2.Add(groupes.First(g => g.nom == nom).contrainte);
        Console.WriteLine("Groupes du noyau seuls : " + s2.Check());

        // Le noyau est-il minimal ? On retire chaque groupe un par un et on re-teste
        Console.WriteLine();
        Console.WriteLine("En retirant un groupe du noyau :");
        foreach (var retire in noyau)
        {
            var s3 = ctx6.MkSolver();
            foreach (var nom in noyau.Where(n => n != retire))
                s3.Add(groupes.First(g => g.nom == nom).contrainte);
            var r = s3.Check();
            var verdict = r == Status.UNSATISFIABLE
                ? "toujours impossible -> ce groupe est inutile"
                : "devient possible -> ce groupe est necessaire";
            Console.WriteLine($"  sans {retire,9} : {r}  ({verdict})");
        }
    }
    else if (statut == Status.UNKNOWN)
        Console.WriteLine("unknown : Z3 n'a pas conclu, on ne peut rien dire");

    // Le noyau n'est pas forcement minimal : Z3 renvoie un ensemble suffisant pour la
    // contradiction, pas le plus petit possible. Le test ci-dessus le montre.
    // unsat = contradiction prouvee ; unknown = aucune conclusion.
}

ExempleGuide6();

Statut : UNSATISFIABLE


Noyau insatisfiable : [domaine; distinct; tetes; equation]


Groupes du noyau seuls : UNSATISFIABLE


En retirant un groupe du noyau :


  sans   domaine : SATISFIABLE  (devient possible -> ce groupe est necessaire)


  sans  distinct : SATISFIABLE  (devient possible -> ce groupe est necessaire)


  sans     tetes : UNSATISFIABLE  (toujours impossible -> ce groupe est inutile)


  sans  equation : SATISFIABLE  (devient possible -> ce groupe est necessaire)


### Exemple guidé 7 — Multiplication par constante

Réalisation de @Will-emma, issue de la PR #28098, portée en Microsoft.Z3.

`ABCD * 4 = DCBA` est modélisé comme une équation positionnelle où le multiplicateur est la constante 4 ; après résolution, la négation du modèle est ajoutée puis re-testée, ce qui établit l'unicité.

In [11]:
// Exemple guide 7 : ABCD * 4 = DCBA (multiplication)
// Portage C# de la realisation de @Will-emma, issue de la PR #28098
void ExempleGuide7()
{
    var ctx7 = new Context();
    var A = ctx7.MkIntConst("A"); var B = ctx7.MkIntConst("B");
    var C = ctx7.MkIntConst("C"); var D = ctx7.MkIntConst("D");
    IntExpr[] ls = { A, B, C, D };

    var s = ctx7.MkSolver();
    foreach (var v in ls)                                              // 1. chiffres 0..9
        s.Add(ctx7.MkAnd(ctx7.MkGe(v, ctx7.MkInt(0)), ctx7.MkLe(v, ctx7.MkInt(9))));
    s.Add(ctx7.MkDistinct(ls));                                        // 2. tous differents
    s.Add(ctx7.MkGt(A, ctx7.MkInt(0)), ctx7.MkGt(D, ctx7.MkInt(0)));   //    tetes non nulles
    ArithExpr Mul7(int w, IntExpr v) => ctx7.MkMul(ctx7.MkInt(w), v);
    var abcd = ctx7.MkAdd(Mul7(1000, A), Mul7(100, B), Mul7(10, C), D);
    var dcba = ctx7.MkAdd(Mul7(1000, D), Mul7(100, C), Mul7(10, B), A);
    s.Add(ctx7.MkEq(ctx7.MkMul(abcd, ctx7.MkInt(4)), dcba));           // 3. equation

    if (s.Check() == Status.SATISFIABLE)
    {
        var m = s.Model;
        Console.WriteLine($"{((IntNum)m.Evaluate(abcd)).Int64} * 4 = {((IntNum)m.Evaluate(dcba)).Int64}");

        // 4. Unicite : on interdit cette solution et on re-teste
        s.Add(ctx7.MkOr(ls.Select(v => ctx7.MkNot(ctx7.MkEq(v, m.Evaluate(v)))).ToArray()));
        if (s.Check() == Status.UNSATISFIABLE)
            Console.WriteLine("unsat : la solution est unique");
        else
            Console.WriteLine("autre solution : " + s.Model);
    }
    else
    {
        Console.WriteLine("Pas de solution");
    }
}

ExempleGuide7();

2178 * 4 = 8712


unsat : la solution est unique


## Exercices à compléter

### Exercice 1 — Énumérer les solutions d'une addition

Modélisez `TWO + TWO = FOUR` (six lettres distinctes, têtes `T` et `F` non nulles) et énumérez **toutes** les solutions avec la technique de l'exemple guidé 2 : après chaque modèle, interdisez-le par une disjonction et relancez. Pour chaque solution trouvée, vérifiez-la par substitution **dans les deux encodages** de l'exemple guidé 4 : l'équation positionnelle d'une part, l'addition posée colonne par colonne avec retenues d'autre part. Le statut terminal distingue l'énumération exhaustive (`UNSATISFIABLE`) d'un arrêt sans preuve (`UNKNOWN`).

In [12]:
// Exercice 1 : enumerer les solutions de TWO + TWO = FOUR
// TODO etudiant :
// 1. Six IntExpr T, W, O, F, U, R, domaine 0..9
// 2. MkDistinct sur les 6 lettres, tetes T > 0 et F > 0
// 3. Equation positionnelle : (100*T + 10*W + O) * 2 == 1000*F + 100*O + 10*U + R
// 4. Boucle d'enumeration : bloquer chaque modele par MkOr(lettre != valeur du modele)
// 5. Pour chaque solution : verification par substitution dans l'equation positionnelle
//    PUIS dans l'addition posee en colonnes avec retenues (r1, r2, r3 entre 0 et 1)
int? nbSolutions = null; // TODO etudiant
Console.WriteLine("Exercice 1 à compléter : énumérez les solutions de TWO + TWO = FOUR");

Exercice 1 à compléter : énumérez les solutions de TWO + TWO = FOUR


### Exercice 2 — Bases paires, bases impaires

Pour chaque base entière fixée de 4 à 12 incluse, modélisez `AB + BC = CA` : trois chiffres distincts, non nuls, avec les poids positionnels propres à la base. Affichez le statut pour chaque base et, lorsqu'il existe, un modèle vérifié par substitution. Le comportement n'est pas uniforme : observez quelles bases acceptent une solution, puis expliquez le pattern — l'identité `b * (A + B - C) = A - B - C` est un bon point de départ. Distinguez absence de solution (`UNSATISFIABLE`) et réponse `UNKNOWN`.

In [13]:
// Exercice 2 : AB + BC = CA dans les bases 4 a 12
// TODO etudiant :
// 1. Trois IntExpr A, B, C
// 2. Pour chaque base b de 4 a 12 : chiffres non nuls de la base, MkDistinct(A, B, C),
//    equation (A*b + B) + (B*b + C) == C*b + A
// 3. Afficher le statut par base et verifier chaque modele trouve par substitution
int? nbSolutionsParBase = null; // TODO etudiant
Console.WriteLine("Exercice 2 à compléter : cryptarithme en bases variables");

Exercice 2 à compléter : cryptarithme en bases variables


### Exercice 3 — Expliquer une autre contradiction

Le célèbre `1089 * 9 = 9801` fonctionne à quatre chiffres. À trois chiffres, modélisez `ABC * 9 = CBA` (chiffres distincts, têtes `A` et `C` non nulles) : le statut attendu est `UNSATISFIABLE`. Comme dans l'exemple guidé 6, étiquetez les groupes de contraintes avec `AssertAndTrack`, extrayez le noyau insatisfiable, vérifiez que les groupes du noyau seuls sont contradictoires, puis retirez chaque groupe un par un pour identifier ceux qui sont nécessaires. Expliquez pourquoi le noyau retourné n'est pas nécessairement minimal, et distinguez `UNSATISFIABLE` de `UNKNOWN`.

In [14]:
// Exercice 3 : expliquer pourquoi ABC * 9 = CBA est impossible
// TODO etudiant :
// 1. Trois IntExpr A, B, C, domaine 0..9, MkDistinct, tetes A > 0 et C > 0
// 2. Groupes etiquetes via AssertAndTrack : domaine, distinct, tetes, equation
// 3. Noyau insatisfiable + verification que les groupes du noyau seuls sont unsat
// 4. Retrait d'un groupe a la fois : ce groupe est necessaire ou inutile ?
bool? resultat = null; // TODO etudiant
Console.WriteLine("Exercice 3 à compléter : noyau insatisfiable de ABC * 9 = CBA");

Exercice 3 à compléter : noyau insatisfiable de ABC * 9 = CBA


### Exercice 4 — Un produit palindromique à cinq chiffres

`2178 * 4 = 8712` est l'unique solution à quatre chiffres (exemple guidé 7). Généralisez à cinq chiffres : modélisez `ABCDE * 4 = EDCBA`, où `ABCDE = 10000*A + 1000*B + 100*C + 10*D + E` et `EDCBA = 10000*E + 1000*D + 100*C + 10*B + A`.

**Étapes** :
1. Cinq `IntExpr` `A, B, C, D, E`, domaine `{0..9}`.
2. `MkDistinct(A, B, C, D, E)`, têtes `A > 0` et `E > 0`.
3. Équation positionnelle : `(10000*A + 1000*B + 100*C + 10*D + E) * 4 == (10000*E + 1000*D + 100*C + 10*B + A)`.
4. Résolvez, puis **prouvez l'unicité** comme dans l'exemple guidé 2 : ajoutez la négation du modèle trouvé et vérifiez que le solveur répond `UNSATISFIABLE`.

**Indice** : le produit `ABCDE * 4` doit rester un nombre à cinq chiffres — la tête `A` est donc très contrainte. Commencez par les unités : `A` est le dernier chiffre de `E * 4`.

In [15]:
// Exercice 4 : ABCDE * 4 = EDCBA (multiplication a cinq chiffres)
// TODO etudiant :
// 1. Cinq IntExpr A, B, C, D, E, domaine 0..9
// 2. MkDistinct(A, B, C, D, E), tetes A > 0 et E > 0
// 3. Equation : (10000*A + 1000*B + 100*C + 10*D + E) * 4
//                == (10000*E + 1000*D + 100*C + 10*B + A)
// 4. Unicite : ajouter la negation du modele trouve et verifier UNSATISFIABLE
long? result = null; // TODO etudiant
Console.WriteLine("Exercice 4 à compléter : modélisez ABCDE * 4 = EDCBA");

Exercice 4 à compléter : modélisez ABCDE * 4 = EDCBA


## Conclusion

Les cryptarithmes illustrent le paradigme déclaratif : on exprime les domaines, les contraintes de distinction et l'équation plutôt que l'algorithme de recherche. Les exemples montrent la recherche d'un témoin, l'énumération des modèles et l'insatisfiabilité d'une multiplication ; les exemples guidés, issus de rendus d'étudiants, prolongent ces méthodes avec des retenues explicites, des bases variables, un noyau insatisfiable et une multiplication par constante. Les exercices suivent le même mouvement : énumération exhaustive d'une addition, bases paires contre bases impaires, impossibilité expliquée par un noyau et produit palindromique à cinq chiffres.

Le pont avec la série sœur Z3.Linq se fait via [`Z3-10-Cryptarithmetic-Python.ipynb`](Z3-10-Cryptarithmetic-Python.ipynb) et les exemples portés entre Microsoft.Z3, pyz3 et Z3.Linq2Z3.

***

> Carnet issu de #27099 (jumeau C# de la série, EPIC #1206 côté Python).